# MIMIC Fidelity Analysis: Real-Data Validation of the Positive-Count Hypothesis

In [22]:
library(duckdb)
library(DBI)
library(dplyr)
library(synthpop)
library(ggplot2)
library(tidyr)

source("config.R")

In [4]:
con <- dbConnect(duckdb())

dbExecute(con, sprintf(
  "CREATE VIEW diagnoses_icd AS SELECT * FROM read_csv_auto('%s/hosp/diagnoses_icd.csv.gz')",
  DATA_DIR))
dbExecute(con, sprintf(
  "CREATE VIEW d_icd_diagnoses AS SELECT * FROM read_csv_auto('%s/hosp/d_icd_diagnoses.csv.gz')",
  DATA_DIR))
dbExecute(con, sprintf(
  "CREATE VIEW icustays AS SELECT * FROM read_csv_auto('%s/icu/icustays.csv.gz')",
  DATA_DIR))
dbExecute(con, sprintf(
  "CREATE VIEW patients AS SELECT * FROM read_csv_auto('%s/hosp/patients.csv.gz')",
  DATA_DIR))
dbExecute(con, sprintf(
  "CREATE VIEW admissions AS SELECT * FROM read_csv_auto('%s/hosp/admissions.csv.gz')",
  DATA_DIR))
dbExecute(con, sprintf(
  "CREATE VIEW labevents AS SELECT * FROM read_csv_auto('%s/hosp/labevents.csv.gz')",
  DATA_DIR))
dbExecute(con, sprintf(
  "CREATE VIEW d_labitems AS SELECT * FROM read_csv_auto('%s/hosp/d_labitems.csv.gz')",
  DATA_DIR))

duckdb keeps downloaded extensions and secrets in a temporary directory:
ℹ /var/folders/5d/pvxmh0mx4f124ss1c7t2ymk00000gn/T//RtmpBt3Y4U/duckdb
This is removed when the R session ends.
• Extensions are re-downloaded each session.
• Secrets are lost.
ℹ Run duckdb(shared_home = TRUE) (or create ~/.duckdb) to keep them (suitable for most users).
ℹ Run duckdb(shared_home = FALSE) to accept the temporary directory (and silence this message).
ℹ See ?duckdb_storage for details and alternatives.


[1] 0

[1] 0

[1] 0

[1] 0

[1] 0

[1] 0

[1] 0

## Comorbidity Screening: Positive Count by Category

In [7]:
comorbidity_counts <- dbGetQuery(con, "
  SELECT
    CASE
      WHEN icd_version = 9  AND icd_code LIKE '428%' THEN 'CHF'
      WHEN icd_version = 10 AND icd_code LIKE 'I50%'  THEN 'CHF'
      WHEN icd_version = 9  AND icd_code LIKE '250%' THEN 'Diabetes'
      WHEN icd_version = 10 AND (icd_code LIKE 'E10%' OR icd_code LIKE 'E11%'
                               OR icd_code LIKE 'E12%' OR icd_code LIKE 'E13%'
                               OR icd_code LIKE 'E14%') THEN 'Diabetes'
      WHEN icd_version = 9  AND icd_code IN ('491','492','494','496') THEN 'COPD'
      WHEN icd_version = 10 AND icd_code LIKE 'J44%' THEN 'COPD'
      WHEN icd_version = 9  AND icd_code LIKE '584%' THEN 'Renal Failure'
      WHEN icd_version = 9  AND icd_code LIKE '585%' THEN 'Renal Failure'
      WHEN icd_version = 9  AND icd_code LIKE '586%' THEN 'Renal Failure'
      WHEN icd_version = 10 AND (icd_code LIKE 'N17%' OR icd_code LIKE 'N18%'
                               OR icd_code LIKE 'N19%') THEN 'Renal Failure'
      WHEN icd_version = 9  AND icd_code LIKE '401%' THEN 'Hypertension'
      WHEN icd_version = 10 AND (icd_code LIKE 'I10%' OR icd_code LIKE 'I11%'
                               OR icd_code LIKE 'I12%' OR icd_code LIKE 'I13%'
                               OR icd_code LIKE 'I15%') THEN 'Hypertension'
      ELSE NULL
    END AS comorbidity,
    COUNT(DISTINCT d.hadm_id) AS positive_count
  FROM diagnoses_icd d
  WHERE d.hadm_id IN (SELECT DISTINCT hadm_id FROM icustays)
    AND comorbidity IS NOT NULL
  GROUP BY comorbidity
  ORDER BY positive_count
")
print(comorbidity_counts)

    comorbidity positive_count
1          COPD           7244
2           CHF          17412
3      Diabetes          19806
4 Renal Failure          25369
5  Hypertension          34979


## Continuous Variable Selection

In [15]:
creatinine_items <- dbGetQuery(con, "
  SELECT itemid, label, fluid, category
  FROM d_labitems
  WHERE LOWER(label) LIKE '%creatinine%'
    AND fluid = 'Blood'
  ORDER BY label
")
print(creatinine_items)

  itemid                   label fluid  category
1  50912              Creatinine Blood Chemistry
2  52546              Creatinine Blood Chemistry
3  52024 Creatinine, Whole Blood Blood Blood Gas


In [16]:
creatinine_coverage <- dbGetQuery(con, "
  WITH total AS (
    SELECT COUNT(*) AS n_stays FROM icustays
  ),
  with_creatinine AS (
    SELECT COUNT(DISTINCT i.stay_id) AS n_with_creatinine
    FROM icustays i
    JOIN labevents l ON i.hadm_id = l.hadm_id
    WHERE l.itemid IN (50912, 52546, 52024)
      AND l.charttime BETWEEN i.intime AND i.intime + INTERVAL 24 HOUR
      AND l.valuenum IS NOT NULL
  )
  SELECT
    total.n_stays,
    with_creatinine.n_with_creatinine,
    ROUND(100.0 * with_creatinine.n_with_creatinine / total.n_stays, 1) AS coverage_pct
  FROM total, with_creatinine
")
print(creatinine_coverage)

  n_stays n_with_creatinine coverage_pct
1   73181             70954           97


In [17]:
creatinine_firstday <- dbGetQuery(con, "
  SELECT
    i.stay_id,
    MAX(l.valuenum) AS creatinine_max
  FROM icustays i
  JOIN labevents l ON i.hadm_id = l.hadm_id
  WHERE l.itemid IN (50912, 52546, 52024)
    AND l.charttime BETWEEN i.intime AND i.intime + INTERVAL 24 HOUR
    AND l.valuenum IS NOT NULL
  GROUP BY i.stay_id
")
str(creatinine_firstday)
summary(creatinine_firstday$creatinine_max)

'data.frame':	70954 obs. of  2 variables:
 $ stay_id       : num  32595996 39668021 33874686 30847801 36540685 ...
 $ creatinine_max: num  2.1 3.5 1.9 0.6 1.7 2.4 0.8 0.4 1.2 2.1 ...


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.700   1.000   1.509   1.500  80.000 

In [19]:
CAP <- quantile(creatinine_firstday$creatinine_max, 0.995, na.rm = TRUE)
creatinine_firstday$creatinine_max_capped <- pmin(creatinine_firstday$creatinine_max, CAP)
print(CAP)

99.5% 
   11 


## Build Flat Table

In [20]:
cohort_base <- dbGetQuery(con, "
  SELECT
    i.stay_id, i.hadm_id, i.subject_id, i.intime, i.outtime, i.los,
    p.gender, p.anchor_age,
    a.admission_type, a.insurance, a.hospital_expire_flag,
    MAX(CASE WHEN (d.icd_version = 9  AND d.icd_code LIKE '428%')
             OR (d.icd_version = 10 AND d.icd_code LIKE 'I50%') THEN 1 ELSE 0 END) AS chf,
    MAX(CASE WHEN (d.icd_version = 9  AND d.icd_code LIKE '250%')
             OR (d.icd_version = 10 AND (d.icd_code LIKE 'E10%' OR d.icd_code LIKE 'E11%'
                                       OR d.icd_code LIKE 'E12%' OR d.icd_code LIKE 'E13%'
                                       OR d.icd_code LIKE 'E14%')) THEN 1 ELSE 0 END) AS diabetes,
    MAX(CASE WHEN (d.icd_version = 9  AND d.icd_code IN ('491','492','494','496'))
             OR (d.icd_version = 10 AND d.icd_code LIKE 'J44%') THEN 1 ELSE 0 END) AS copd,
    MAX(CASE WHEN (d.icd_version = 9  AND (d.icd_code LIKE '584%' OR d.icd_code LIKE '585%' OR d.icd_code LIKE '586%'))
             OR (d.icd_version = 10 AND (d.icd_code LIKE 'N17%' OR d.icd_code LIKE 'N18%'
                                       OR d.icd_code LIKE 'N19%')) THEN 1 ELSE 0 END) AS renal_failure,
    MAX(CASE WHEN (d.icd_version = 9  AND d.icd_code LIKE '401%')
             OR (d.icd_version = 10 AND (d.icd_code LIKE 'I10%' OR d.icd_code LIKE 'I11%'
                                       OR d.icd_code LIKE 'I12%' OR d.icd_code LIKE 'I13%'
                                       OR d.icd_code LIKE 'I15%')) THEN 1 ELSE 0 END) AS hypertension
  FROM icustays i
  LEFT JOIN patients p      ON i.subject_id = p.subject_id
  LEFT JOIN admissions a    ON i.hadm_id = a.hadm_id
  LEFT JOIN diagnoses_icd d ON i.hadm_id = d.hadm_id
  GROUP BY i.stay_id, i.hadm_id, i.subject_id, i.intime, i.outtime, i.los,
           p.gender, p.anchor_age, a.admission_type, a.insurance, a.hospital_expire_flag
")
str(cohort_base)

'data.frame':	73181 obs. of  16 variables:
 $ stay_id             : num  33691075 35589504 30258557 38508091 38432876 ...
 $ hadm_id             : num  27282308 25941781 25941781 27279120 25067748 ...
 $ subject_id          : num  11804719 11805066 11805066 11810079 11810885 ...
 $ intime              : POSIXct, format: "2122-11-16 16:01:59" "2185-05-05 06:28:11" ...
 $ outtime             : POSIXct, format: "2122-11-22 17:48:59" "2185-05-07 17:53:53" ...
 $ los                 : num  6.074 2.476 13.588 1.09 0.616 ...
 $ gender              : chr  "M" "F" "F" "M" ...
 $ anchor_age          : num  47 68 68 67 83 62 63 69 60 44 ...
 $ admission_type      : chr  "EW EMER." "OBSERVATION ADMIT" "OBSERVATION ADMIT" "OBSERVATION ADMIT" ...
 $ insurance           : chr  "Medicare" "Medicare" "Medicare" "Other" ...
 $ hospital_expire_flag: num  0 1 1 0 0 0 0 0 0 0 ...
 $ chf                 : int  0 0 0 0 0 0 0 0 0 1 ...
 $ diabetes            : int  0 0 0 1 1 0 0 1 0 1 ...
 $ copd             

In [21]:
flat_table <- cohort_base %>%
  left_join(
    creatinine_firstday %>% select(stay_id, creatinine_max_capped),
    by = "stay_id"
  )

str(flat_table)

'data.frame':	73181 obs. of  17 variables:
 $ stay_id              : num  33691075 35589504 30258557 38508091 38432876 ...
 $ hadm_id              : num  27282308 25941781 25941781 27279120 25067748 ...
 $ subject_id           : num  11804719 11805066 11805066 11810079 11810885 ...
 $ intime               : POSIXct, format: "2122-11-16 16:01:59" "2185-05-05 06:28:11" ...
 $ outtime              : POSIXct, format: "2122-11-22 17:48:59" "2185-05-07 17:53:53" ...
 $ los                  : num  6.074 2.476 13.588 1.09 0.616 ...
 $ gender               : chr  "M" "F" "F" "M" ...
 $ anchor_age           : num  47 68 68 67 83 62 63 69 60 44 ...
 $ admission_type       : chr  "EW EMER." "OBSERVATION ADMIT" "OBSERVATION ADMIT" "OBSERVATION ADMIT" ...
 $ insurance            : chr  "Medicare" "Medicare" "Medicare" "Other" ...
 $ hospital_expire_flag : num  0 1 1 0 0 0 0 0 0 0 ...
 $ chf                  : int  0 0 0 0 0 0 0 0 0 1 ...
 $ diabetes             : int  0 0 0 1 1 0 0 1 0 1 ...
 $ copd